# Stance coding with `bendavidsteel/Qwen3.5-4B-stance-detection`

Model, prompt and `classify()` follow the model card. **Target** = the episode's `Choice` (if any) plus the episode's first post (earliest `created_at`).
Labels `favor` / `against` / `neutral` become valence `+1` / `-1` / `0`. Each result is appended to `stance_results.jsonl` as it finishes, so an interruption loses nothing; just re-run.

In [1]:
%pip install -q -U torch transformers pymongo pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, os, torch, pandas as pd
from pymongo import MongoClient

N = 20            # e.g. 20 for a quick trial run
OUT = "stance_results.jsonl"

db = MongoClient(json.load(open("config.json"))["mongo_uri"])["consensus"]
choice = {d["Episode #"]: d.get("Choice") for d in db["python_source_map"].find({}, {"Episode #": 1, "Choice": 1})}
comments = pd.DataFrame(db["python_source_map_comments"].find({}, {"Episode #": 1, "text": 1, "created_at": 1})).dropna(subset=["text"])
comments = comments.sort_values("created_at")
first = comments.groupby("Episode #").head(1)                      # first post of each episode
post = dict(zip(first["Episode #"], first["text"]))
todo = comments[~comments["_id"].isin(first["_id"]) & comments["Episode #"].isin(post)]
if N:
    todo = todo.head(N)


def target_for(ep):
    p = " ".join(str(post[ep]).split())[:600]
    c = choice.get(ep)
    return f"{c.strip()}: {p}" if isinstance(c, str) and c.strip() else p


print(f"{len(todo)} comments to code | {len(post)} episodes")

20 comments to code | 129 episodes


In [3]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

repo = "bendavidsteel/Qwen3.5-4B-stance-detection"
tokenizer = AutoTokenizer.from_pretrained(repo)
model = AutoModelForSequenceClassification.from_pretrained(repo, dtype=torch.bfloat16, device_map="mps")
model.eval()

PROMPT = (
    "For the following text, determine whether the author's stance is in favor of, "
    "against, or neutral toward the target: '{target}'. Consider the language used, "
    "any explicit statements of position, and contextual clues that suggest the "
    "author's stance. Answer only with 'in favor', 'against', or 'neutral'. \n\n"
    "Text: '{text}'"
)


def classify(text, target):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": PROMPT.format(target=target, text=text)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,
        truncation=True,
        max_length=2048,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0]
    probs = logits.softmax(-1)
    return model.config.id2label[int(probs.argmax())], probs.tolist()

/Users/Michael/repos/consensus/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1016 12:50:20.847000 1694 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 427/427 [00:06<00:00, 66.62it/s]


In [4]:
done = {json.loads(l)["_id"] for l in open(OUT)} if os.path.exists(OUT) else set()
with open(OUT, "a") as f:
    for n, (_, r) in enumerate(todo.iterrows()):
        if str(r["_id"]) in done:
            continue
        c = choice.get(r["Episode #"])
        label, probs = classify(" ".join(r["text"].split())[:3000], target_for(r["Episode #"]))
        f.write(json.dumps({"_id": str(r["_id"]), "label": label, "probs": probs,
                            "choice": c if isinstance(c, str) and c.strip() else None}) + "\n")
        f.flush()
        if n % 25 == 0:
            torch.mps.empty_cache()
            print(f"{n}/{len(todo)}", flush=True)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


0/20


In [5]:
from bson import ObjectId
from pymongo import UpdateOne

WRITE = False       # True = overwrite valence and valence_meta
val = {"favor": 1, "against": -1, "neutral": 0}
names = [model.config.id2label[i] for i in range(3)]
ops = []
for l in open(OUT):
    d = json.loads(l)
    ops.append(UpdateOne({"_id": ObjectId(d["_id"])}, {"$set": {
        "valence": val[d["label"]],
        "valence_meta": {"label": d["label"], "probs": dict(zip(names, d["probs"])), "target": d["choice"], "model": repo}}}))
print(f"{len(ops)} updates", "(written)" if WRITE else "(not written; set WRITE = True)")
if WRITE:
    db["python_source_map_comments"].bulk_write(ops)

20 updates (not written; set WRITE = True)
